In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

# Read the locally downloaded dataset
local_csv = "datascience_salaries.csv"
if not os.path.exists(local_csv):
    raise FileNotFoundError(f"Local dataset not found: {local_csv}")

df = pd.read_csv(local_csv)
print("Loaded local dataset shape:", df.shape)
print(df.head())
print("\nColumns:", df.columns.tolist())

# Normalize the salary column using Min-Max scaling
salary_min = df["salary"].min()
salary_max = df["salary"].max()
df["salary_minmax"] = (df["salary"] - salary_min) / (salary_max - salary_min)
print("\nSalary normalization complete. Sample values:")
print(df[["salary", "salary_minmax"]].head())

# Prepare features for dimensionality reduction
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if "salary" in numeric_cols:
    numeric_cols.remove("salary")
if "salary_minmax" in numeric_cols:
    numeric_cols.remove("salary_minmax")

categorical_cols = df.select_dtypes(include=[object]).columns.tolist()
categorical_cols = [col for col in categorical_cols if col != "experience_level"]

print("\nNumeric columns used for PCA:", numeric_cols)
print("Categorical columns converted for PCA:", categorical_cols)

feature_df = df[numeric_cols].copy()
if categorical_cols:
    feature_df = pd.concat([feature_df, pd.get_dummies(df[categorical_cols], drop_first=True)], axis=1)

# Standardize features before PCA
scaler = MinMaxScaler()
feature_scaled = scaler.fit_transform(feature_df)

pca = PCA(n_components=2)
pca_components = pca.fit_transform(feature_scaled)

pca_df = pd.DataFrame(pca_components, columns=["PC1", "PC2"])
print("\nPCA complete. Explained variance ratios:")
print(pd.Series(pca.explained_variance_ratio_, index=["PC1", "PC2"]))
print(pca_df.head())

# Group by experience level and calculate average and median salary
if "experience_level" not in df.columns:
    raise KeyError("Column 'experience_level' not found in the dataset.")

salary_by_experience = df.groupby("experience_level")["salary"].agg(["mean", "median"]).rename(columns={"mean": "average_salary", "median": "median_salary"})
print("\nAverage and median salary by experience level:")
print(salary_by_experience)
